In [ ]:
!pip install -q -U transformers datasets sentence-transformers pandas accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 146.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 

In [ ]:
from google.colab import userdata, drive
from huggingface_hub import login
import os

PROJECT_DIR = '/content/'
print(f"Data will be safely saved to: {PROJECT_DIR}")

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

Data will be safely saved to: /content/


In [ ]:
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
from google.colab import userdata, drive
from huggingface_hub import login
import os

# ==========================================
# 1. SETUP & AUTHENTICATION
# ==========================================

MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
EMBEDDER_ID = "all-MiniLM-L6-v2"
OUTPUT_FILE = f"{PROJECT_DIR}/medhallu_lora_training_data_batch.csv"
BATCH_SIZE = 16 # <--- The A100 Speed Secret

# ==========================================
# 2. LOAD MODELS (GPU NATIVE)
# ==========================================
print("Loading Llama-3 into VRAM (bfloat16)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="cuda:0",
    torch_dtype=torch.bfloat16
)

print("Loading Sentence Transformer...")
grader_model = SentenceTransformer(EMBEDDER_ID, device="cuda:0")

# ==========================================
# 3. PREPARE DATASET
# ==========================================
print("Loading MedHallu Dataset...")
dataset = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_artificial", split="train")
df = pd.DataFrame(dataset)

prompts = []
for idx, row in df.iterrows():
    context = row['Knowledge'] if isinstance(row['Knowledge'], str) else " ".join(row['Knowledge'])
    prompts.append(f"Context: {context}\n\nQuestion: {row['Question']}\n\nProvide a concise medical answer:\n")

df['prompt'] = prompts

# ==========================================
# 4. BATCH GENERATION LOOP
# ==========================================
print(f"\nStarting Batch Generation ({BATCH_SIZE} at a time)...")
generated_answers = []

# Process in chunks to keep the GPU fed
for i in tqdm(range(0, len(df), BATCH_SIZE)):
    batch_prompts = df['prompt'].iloc[i:i+BATCH_SIZE].tolist()

    # Tokenize the entire batch
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(model.device)

    with torch.no_grad():
        gen_outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False
        )

    # Decode the batch and strip out the prompt
    for j, output in enumerate(gen_outputs):
        input_len = inputs['input_ids'][j].shape[0]
        # Slice the tensor to grab only the newly generated tokens
        new_tokens = output[input_len:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        generated_answers.append(answer)

df['generated_answer'] = generated_answers

# Clear the LLM from memory to make room for grading
del model
torch.cuda.empty_cache()

# ==========================================
# 5. VECTORIZED GRADING
# ==========================================
print("\nEmbedding sentences for batch grading...")
# We do all 9,000 at once using the fast PyTorch backend
emb_gen = grader_model.encode(df['generated_answer'].tolist(), batch_size=256, convert_to_tensor=True)
emb_gt = grader_model.encode(df['Ground Truth'].tolist(), batch_size=256, convert_to_tensor=True)
emb_fake = grader_model.encode(df['Hallucinated Answer'].tolist(), batch_size=256, convert_to_tensor=True)

print("Calculating semantic similarities...")
sim_to_gt = torch.nn.functional.cosine_similarity(emb_gen, emb_gt, dim=1)
sim_to_fake = torch.nn.functional.cosine_similarity(emb_gen, emb_fake, dim=1)

# Generate Labels: 0 for Safe, 1 for Hallucination
labels = (sim_to_gt < sim_to_fake).to(torch.int).cpu().numpy()

# ==========================================
# 6. SAVE
# ==========================================
df_out = pd.DataFrame({
    "prompt": df['prompt'],
    "label": labels
})
df_out.to_csv(OUTPUT_FILE, index=False)

unique, counts = torch.unique(torch.tensor(labels), return_counts=True)
dist = dict(zip(unique.tolist(), counts.tolist()))

print("="*50)
print(f"✅ Phase 1 Complete (Batch Mode): Saved to {OUTPUT_FILE}")
print(f"📊 Dataset Distribution (0=Safe, 1=Hallucination): {dist}")
print("="*50)

Loading Llama-3 into VRAM (bfloat16)...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/9000 [00:00<?, ? examples/s]


Starting Batch Generation (16 at a time)...


  0%|          | 0/563 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Embedding sentences for batch grading...
Calculating semantic similarities...
✅ Phase 1 Complete (Batch Mode): Saved to /content//medhallu_lora_training_data_batch.csv
📊 Dataset Distribution (0=Safe, 1=Hallucination): {0: 2393, 1: 6607}
